In [1]:
with open('./hgnc_complete_set.txt', 'r') as fopen:
    lines = fopen.readlines()

mapping = {}
for line in lines[1:]:  # skip header
    HGNC_id, symbol = line.strip().split('\t')[:2]
    mapping[HGNC_id] = symbol

mapping["HGNC:49667"] = "ABALON"
mapping["HGNC:58098"] = "TSNAX-DT"

In [2]:
import json
from pathlib import Path

# --- config ---
vocab_path = Path("../model/assets/condtech_coarse/vocab.json")
out_path = vocab_path.with_name("vocab_symbols.json")

# --- load vocab ---
with open(vocab_path) as f:
    vocab = json.load(f)

# Identify non-HGNC special tokens (keep as-is)
def is_hgnc_key(k: str) -> bool:
    return isinstance(k, str) and k.startswith("HGNC:") and k[5:].isdigit()

special_items = {k: v for k, v in vocab.items() if not is_hgnc_key(k)}
hgnc_items = {k: v for k, v in vocab.items() if is_hgnc_key(k)}

# Prepare numeric HGNC accessions (strip 'HGNC:')
hgnc_ids = [k for k in hgnc_items.keys()]  # e.g. "4788"

# Build final dict: symbols & specials preserved
symbol_vocab = {}

# 1) specials (unchanged)
symbol_vocab.update(special_items)

# 2) HGNC keys → gene symbols (keep original indices)
missing = []
for hgnc_key, idx in hgnc_items.items():
    symbol = mapping.get(hgnc_key)
    if symbol:
        symbol_vocab[symbol] = idx
    else:
        missing.append(hgnc_key)  # no mapping returned

# Save
with open(out_path, "w") as f:
    json.dump(symbol_vocab, f, indent=2, sort_keys=True)

print(f"Saved symbol-based vocab to {out_path}")
if missing:
    print(f"WARNING: {len(missing)} HGNC IDs had no symbol mapping (kept out). "
          f"Examples: {missing[:10]}")


Saved symbol-based vocab to ../model/assets/condtech_coarse/vocab_symbols.json
